<div style="background-color: #04D7FD; padding: 20px; text-align: left;">
    <h1 style="color: #000000; font-size: 36px; margin: 0;">Data Processing for RAG with Data Prep Kit (RAY)</h1>
    
</div>


## Before Running the notebook

Please complete [setting up python dev environment](./setup-python-dev-env.md)

## Overview

This notebook will process PDF documents as part of RAG pipeline

![](media/rag-overview-2.png)

This notebook will perform steps 1, 2 and 3 in RAG pipeline.

Here are the processing steps:

Here are the processing steps:

- **pdf2parquet** : Extract text (in markdown format) from PDF and store them as parquet files
- **Exact Dedup**: Documents with exact content are filtered out
- **Chunk documents**: Split the PDFs into 'meaningful sections' (paragraphs, sentences ..etc)
- **Text encoder**: Convert chunks into vectors using embedding models

## Step-1: Configuration

In [1]:
## setup path to utils folder
import sys
sys.path.append('../utils')

In [2]:
import os
from my_config import MY_CONFIG

## RAY CONFIGURATION
num_cpus_available =  os.cpu_count()
# print (num_cpus_available)
# MY_CONFIG.RAY_NUM_CPUS = num_cpus_available // 2  ## use half the available cores for processing
MY_CONFIG.RAY_NUM_CPUS =  0.5
MY_CONFIG.RAY_MEMORY_GB = 2  # GB
# MY_CONFIG.RAY_RUNTIME_WORKERS = num_cpus_available // 3
MY_CONFIG.RAY_RUNTIME_WORKERS = 2

print (f"Ray configuration: CPUs={MY_CONFIG.RAY_NUM_CPUS},   memory={MY_CONFIG.RAY_MEMORY_GB} GB,  workers={MY_CONFIG.RAY_RUNTIME_WORKERS}")

Ray configuration: CPUs=0.5,   memory=2 GB,  workers=2


## Step-2:  Data

We will use white papers  about LLMs.  

- [Granite Code Models](https://arxiv.org/abs/2405.04324)
- [Attention is all you need](https://arxiv.org/abs/1706.03762)

You can of course substite your own data below

### 2.1 - Download data

In [3]:
import os, sys
import shutil
from file_utils import download_file

shutil.rmtree(MY_CONFIG.INPUT_DATA_DIR, ignore_errors=True)
shutil.os.makedirs(MY_CONFIG.INPUT_DATA_DIR, exist_ok=True)
print ("✅ Cleared input directory")
 
download_file (url = 'https://arxiv.org/pdf/1706.03762', local_file = os.path.join(MY_CONFIG.INPUT_DATA_DIR, 'attention.pdf' ))
download_file (url = 'https://arxiv.org/pdf/2405.04324', local_file = os.path.join(MY_CONFIG.INPUT_DATA_DIR, 'granite.pdf' ))
download_file (url = 'https://arxiv.org/pdf/2405.04324', local_file = os.path.join(MY_CONFIG.INPUT_DATA_DIR, 'granite2.pdf' )) # duplicate


✅ Cleared input directory

input/attention.pdf (2.22 MB) downloaded successfully.

input/granite.pdf (1.27 MB) downloaded successfully.

input/granite2.pdf (1.27 MB) downloaded successfully.


### 2.2  - Set input/output path variables for the pipeline

In [4]:
import os, sys
import shutil

if not os.path.exists(MY_CONFIG.INPUT_DATA_DIR ):
    raise Exception (f"❌ Input folder MY_CONFIG.INPUT_DATA_DIR = '{MY_CONFIG.INPUT_DATA_DIR}' not found")

output_parquet_dir = os.path.join (MY_CONFIG.OUTPUT_FOLDER, '01_parquet_out')
output_exact_dedupe_dir = os.path.join (MY_CONFIG.OUTPUT_FOLDER, '02_dedupe_out')
output_chunk_dir = os.path.join (MY_CONFIG.OUTPUT_FOLDER, '03_chunk_out')
output_embeddings_dir = os.path.join (MY_CONFIG.OUTPUT_FOLDER, '04_embeddings_out')


## clear output folder
shutil.rmtree(MY_CONFIG.OUTPUT_FOLDER, ignore_errors=True)
shutil.os.makedirs(MY_CONFIG.OUTPUT_FOLDER, exist_ok=True)

print ("✅ Cleared output directory")

✅ Cleared output directory


## Step-3: pdf2parquet -  Convert data from PDF to Parquet

This step is reading the input folder containing all PDF files and ingest them in a parquet table using the [Docling package](https://github.com/DS4SD/docling).
The documents are converted into a JSON format which allows to easily chunk it in the later steps.



### 3.1 -  Execute 

In [5]:
%%time 

from dpk_pdf2parquet.ray.transform import Pdf2Parquet
from data_processing.utils import GB
from dpk_pdf2parquet.transform import pdf2parquet_contents_types

STAGE = 1 
print (f"🏃🏼 STAGE-{STAGE}: Processing input='{MY_CONFIG.INPUT_DATA_DIR}' --> output='{output_parquet_dir}'\n", flush=True)

result = Pdf2Parquet(input_folder= MY_CONFIG.INPUT_DATA_DIR,
                    output_folder= output_parquet_dir, 
                    data_files_to_use=['.pdf'],
                    pdf2parquet_contents_type=pdf2parquet_contents_types.MARKDOWN,
                    # pdf2parquet_contents_type=pdf2parquet_contents_types.JSON,
                    
                    ## runtime options
                    # run_locally= True,
                    # num_cpus= MY_CONFIG.RAY_NUM_CPUS,
                    # memory= MY_CONFIG.RAY_MEMORY_GB * GB,
                    # runtime_num_workers = MY_CONFIG.RAY_RUNTIME_WORKERS,

                    ## debug
                    run_locally= True,
                    num_cpus=  1, 
                    memory= MY_CONFIG.RAY_MEMORY_GB * GB, 
                    runtime_num_workers = 1, ## Note: has to be one for this particular job, to prevent race condition when downloading models!
               ).transform()

if result == 0:
    print (f"✅ Stage:{STAGE} completed successfully")
else:
    raise Exception (f"❌ Stage:{STAGE}  failed")

🏃🏼 STAGE-1: Processing input='input' --> output='output/01_parquet_out'



00:52:31 INFO - pdf2parquet parameters are : {'batch_size': -1, 'artifacts_path': None, 'contents_type': <pdf2parquet_contents_types.MARKDOWN: 'text/markdown'>, 'do_table_structure': True, 'do_ocr': True, 'ocr_engine': <pdf2parquet_ocr_engine.EASYOCR: 'easyocr'>, 'bitmap_area_threshold': 0.05, 'pdf_backend': <pdf2parquet_pdf_backend.DLPARSE_V2: 'dlparse_v2'>, 'double_precision': 8}
00:52:31 INFO - pipeline id pipeline_id
00:52:31 INFO - code location None
00:52:31 INFO - number of workers 1 worker options {'num_cpus': 1, 'memory': 2147483648, 'max_restarts': -1}
00:52:31 INFO - actor creation delay 0
00:52:31 INFO - job details {'job category': 'preprocessing', 'job name': 'pdf2parquet', 'job type': 'ray', 'job id': 'job_id'}
00:52:31 INFO - data factory data_ is using local data access: input_folder - input output_folder - output/01_parquet_out
00:52:31 INFO - data factory data_ max_files -1, n_sample -1
00:52:31 INFO - data factory data_ Not using data sets, checkpointing False, max 

✅ Stage:1 completed successfully
CPU times: user 5.21 s, sys: 1.01 s, total: 6.21 s
Wall time: 7min 11s


### 3.2 - Inspect Generated output

Here we should see one entry per input file processed

In [6]:
from file_utils import read_parquet_files_as_df

output_df = read_parquet_files_as_df(output_parquet_dir)
# print ("Output dimensions (rows x columns)= ", output_df.shape)
output_df.head(5)
## To display certain columns
#parquet_df[['column1', 'column2', 'column3']].head(5)

Successfully read 3 parquet files with 3 total rows


,filename,contents,num_pages,num_tables,num_doc_elements,document_id,document_hash,ext,hash,size,date_acquired,pdf_convert_time,source_filename
0,attention.pdf,"Provided proper attribution is provided, Googl...",15,6,436,12bf79a1-c716-48df-aaa5-5fbc67b513c9,2949302674760005271,pdf,5bbff6b2beac5f09f368a913932212e8d57953b3e3e86d...,45855,2025-03-14T00:53:42.738448,58.294557,attention.pdf
1,granite2.pdf,## Granite Code Models: A Family of Open Found...,28,21,324,d2069e80-be98-4d11-9994-8b98d1d13dc6,3127757990743433032,pdf,96e227c7a23adc6a98590182a29641d1d8ba558847d718...,136446,2025-03-14T00:59:26.979033,172.002515,granite2.pdf
2,granite.pdf,## Granite Code Models: A Family of Open Found...,28,21,324,21000182-6d62-46d3-a810-c4690afb022d,3127757990743433032,pdf,96e227c7a23adc6a98590182a29641d1d8ba558847d718...,136446,2025-03-14T00:56:34.927138,172.116436,granite.pdf


## Step-4: Eliminate Duplicate Documents

We have 2 duplicate documnets here : `granite.pdf` and `granite2.pdf`.

Note how the `hash` for these documents are same.

We are going to perform **de-dupe**

On the content of each document, a SHA256 hash is computed, followed by de-duplication of record having identical hashes.

[Dedupe transform documentation](https://github.com/IBM/data-prep-kit/blob/dev/transforms/universal/ededup/README.md)

### 4.1 - Execute 

In [7]:
%%time 

from dpk_ededup.ray.transform import Ededup

STAGE = 2
print (f"🏃🏼 STAGE-{STAGE}: Processing input='{output_parquet_dir}' --> output='{output_exact_dedupe_dir}'\n", flush=True)

result = Ededup(input_folder=output_parquet_dir,
                output_folder=output_exact_dedupe_dir,
                ededup_hash_cpu= 0.5,
                ededup_num_hashes= 2,
                ededup_doc_column="contents",
                ededup_doc_id_column="document_id",
                
                ## runtime options
                run_locally= True,
                num_cpus= MY_CONFIG.RAY_NUM_CPUS,
                memory= MY_CONFIG.RAY_MEMORY_GB * GB,
                runtime_num_workers = MY_CONFIG.RAY_RUNTIME_WORKERS,
    ).transform()

if result == 0:
    print (f"✅ Stage:{STAGE} completed successfully")
else:
    raise Exception (f"❌ Stage:{STAGE}  failed")

🏃🏼 STAGE-2: Processing input='output/01_parquet_out' --> output='output/02_dedupe_out'



00:59:38 INFO - exact dedup params are {'doc_column': 'contents', 'doc_id_column': 'document_id', 'use_snapshot': False, 'snapshot_directory': None, 'hash_cpu': 0.5, 'num_hashes': 2}
00:59:38 INFO - pipeline id pipeline_id
00:59:38 INFO - code location None
00:59:38 INFO - number of workers 2 worker options {'num_cpus': 0.5, 'memory': 2147483648, 'max_restarts': -1}
00:59:38 INFO - actor creation delay 0
00:59:38 INFO - job details {'job category': 'preprocessing', 'job name': 'ededup', 'job type': 'ray', 'job id': 'job_id'}
00:59:38 INFO - data factory data_ is using local data access: input_folder - output/01_parquet_out output_folder - output/02_dedupe_out
00:59:38 INFO - data factory data_ max_files -1, n_sample -1
00:59:38 INFO - data factory data_ Not using data sets, checkpointing False, max files -1, random samples -1, files to use ['.parquet'], files to checkpoint ['.parquet']
00:59:38 INFO - Running locally
2025-03-14 00:59:39,880	INFO worker.py:1777 -- Started a local Ray in

✅ Stage:2 completed successfully
CPU times: user 117 ms, sys: 160 ms, total: 277 ms
Wall time: 15.7 s


### 4.2 - Inspect Generated output

We would see 2 documents: `attention.pdf`  and `granite.pdf`.  The duplicate `granite.pdf` has been filtered out!

In [8]:
from file_utils import read_parquet_files_as_df

input_df = read_parquet_files_as_df(output_parquet_dir)
output_df = read_parquet_files_as_df(output_exact_dedupe_dir)

# print ("Input data dimensions (rows x columns)= ", input_df.shape)
# print ("Output data dimensions (rows x columns)= ", output_df.shape)
print (f"Input files before exact dedupe : {input_df.shape[0]:,}")
print (f"Output files after exact dedupe : {output_df.shape[0]:,}")
print ("Duplicate files removed :  ", (input_df.shape[0] - output_df.shape[0]))

output_df.sample(min(3, output_df.shape[0]))

Successfully read 3 parquet files with 3 total rows
Successfully read 2 parquet files with 2 total rows
Input files before exact dedupe : 3
Output files after exact dedupe : 2
Duplicate files removed :   1


,filename,contents,num_pages,num_tables,num_doc_elements,document_id,document_hash,ext,hash,size,date_acquired,pdf_convert_time,source_filename,removed
1,granite.pdf,## Granite Code Models: A Family of Open Found...,28,21,324,21000182-6d62-46d3-a810-c4690afb022d,3127757990743433032,pdf,96e227c7a23adc6a98590182a29641d1d8ba558847d718...,136446,2025-03-14T00:56:34.927138,172.116436,granite.pdf,[]
0,attention.pdf,"Provided proper attribution is provided, Googl...",15,6,436,12bf79a1-c716-48df-aaa5-5fbc67b513c9,2949302674760005271,pdf,5bbff6b2beac5f09f368a913932212e8d57953b3e3e86d...,45855,2025-03-14T00:53:42.738448,58.294557,attention.pdf,[]


##  Step-5: Doc chunks

Split the documents in chunks.

[Chunking transform documentation](https://github.com/IBM/data-prep-kit/blob/dev/transforms/language/doc_chunk/README.md)

**Experiment with chunking size to find the setting that works best for your documents**

### 5.1 -  Execute 

In [9]:
%%time

from dpk_doc_chunk.ray.transform import DocChunk
from data_processing.utils import GB

STAGE = 3
print (f"🏃🏼 STAGE-{STAGE}: Processing input='{output_exact_dedupe_dir}' --> output='{output_chunk_dir}'\n", flush=True)

result = DocChunk(input_folder=output_exact_dedupe_dir,
                        output_folder=output_chunk_dir,
                        doc_chunk_chunking_type= "li_markdown",

                        ## runtime options
                        run_locally= True,
                        num_cpus= MY_CONFIG.RAY_NUM_CPUS,
                        memory= MY_CONFIG.RAY_MEMORY_GB * GB,
                        runtime_num_workers = MY_CONFIG.RAY_RUNTIME_WORKERS,
                        ).transform()

if result == 0:
    print (f"✅ Stage:{STAGE} completed successfully")
else:
    raise Exception (f"❌ Stage:{STAGE}  failed")

🏃🏼 STAGE-3: Processing input='output/02_dedupe_out' --> output='output/03_chunk_out'



00:59:55 INFO - doc_chunk parameters are : {'chunking_type': 'li_markdown', 'content_column_name': 'contents', 'doc_id_column_name': 'document_id', 'output_chunk_column_name': 'contents', 'output_source_doc_id_column_name': 'source_document_id', 'output_jsonpath_column_name': 'doc_jsonpath', 'output_pageno_column_name': 'page_number', 'output_bbox_column_name': 'bbox', 'chunk_size_tokens': 128, 'chunk_overlap_tokens': 30, 'dl_min_chunk_len': None}
00:59:55 INFO - pipeline id pipeline_id
00:59:55 INFO - code location None
00:59:55 INFO - number of workers 2 worker options {'num_cpus': 0.5, 'memory': 2147483648, 'max_restarts': -1}
00:59:55 INFO - actor creation delay 0
00:59:55 INFO - job details {'job category': 'preprocessing', 'job name': 'doc_chunk', 'job type': 'ray', 'job id': 'job_id'}
00:59:55 INFO - data factory data_ is using local data access: input_folder - output/02_dedupe_out output_folder - output/03_chunk_out
00:59:55 INFO - data factory data_ max_files -1, n_sample -1
0

✅ Stage:3 completed successfully
CPU times: user 860 ms, sys: 314 ms, total: 1.17 s
Wall time: 18.5 s


### 5.2 - Inspect Generated output

We would see documents are split into many chunks

In [10]:
from file_utils import read_parquet_files_as_df

input_df = read_parquet_files_as_df(output_exact_dedupe_dir)  ## for debug purposes
output_df = read_parquet_files_as_df(output_chunk_dir)

print (f"Files processed : {input_df.shape[0]:,}")
print (f"Chunks created : {output_df.shape[0]:,}")

# print ("Input data dimensions (rows x columns)= ", input_df.shape)
# print ("Output data dimensions (rows x columns)= ", output_df.shape)

output_df.sample(min(3, output_df.shape[0]))

Successfully read 2 parquet files with 2 total rows
Successfully read 2 parquet files with 61 total rows
Files processed : 2
Chunks created : 61


,filename,num_pages,num_tables,num_doc_elements,document_hash,ext,hash,size,date_acquired,pdf_convert_time,source_filename,removed,source_document_id,contents,document_id
0,attention.pdf,15,6,436,2949302674760005271,pdf,5bbff6b2beac5f09f368a913932212e8d57953b3e3e86d...,45855,2025-03-14T00:53:42.738448,58.294557,attention.pdf,[],12bf79a1-c716-48df-aaa5-5fbc67b513c9,"Provided proper attribution is provided, Googl...",cb3a9de8b5457dde64115f2fcd6e5b4093b228499af4ac...
28,granite.pdf,28,21,324,3127757990743433032,pdf,96e227c7a23adc6a98590182a29641d1d8ba558847d718...,136446,2025-03-14T00:56:34.927138,172.116436,granite.pdf,[],21000182-6d62-46d3-a810-c4690afb022d,## Granite Code Models: A Family of Open Found...,4ba39540df65ca93d9dc3026e7ffcea2f949ce9815229c...
7,attention.pdf,15,6,436,2949302674760005271,pdf,5bbff6b2beac5f09f368a913932212e8d57953b3e3e86d...,45855,2025-03-14T00:53:42.738448,58.294557,attention.pdf,[],12bf79a1-c716-48df-aaa5-5fbc67b513c9,## 3.2 Attention\n\nAn attention function can ...,bac7e7f7e0121639aa67e546cc42f16c1b1b3a4c5083f8...


## Step-6:   Calculate Embeddings for Chunks

we will calculate embeddings for each chunk using an open source embedding model

[Embeddings / Text Encoder documentation](https://github.com/IBM/data-prep-kit/blob/dev/transforms/language/text_encoder/README.md)

### 6.1 - Execute

In [11]:
%%time 

from dpk_text_encoder.ray.transform import TextEncoder
from data_processing.utils import GB

STAGE  = 4
print (f"🏃🏼 STAGE-{STAGE}: Processing input='{output_chunk_dir}' --> output='{output_embeddings_dir}'\n", flush=True)

result = TextEncoder(input_folder= output_chunk_dir, 
               output_folder= output_embeddings_dir, 
               text_encoder_model_name = MY_CONFIG.EMBEDDING_MODEL,
               
               ## runtime options
               run_locally= True,
               num_cpus= MY_CONFIG.RAY_NUM_CPUS,
               memory= MY_CONFIG.RAY_MEMORY_GB * GB,
               runtime_num_workers = MY_CONFIG.RAY_RUNTIME_WORKERS,
               ).transform()

if result == 0:
    print (f"✅ Stage:{STAGE} completed successfully")
else:
    raise Exception (f"❌ Stage:{STAGE}  failed")

🏃🏼 STAGE-4: Processing input='output/03_chunk_out' --> output='output/04_embeddings_out'



01:00:13 INFO - text_encoder parameters are : {'content_column_name': 'contents', 'output_embeddings_column_name': 'embeddings', 'model_name': 'ibm-granite/granite-embedding-30m-english'}
01:00:13 INFO - pipeline id pipeline_id
01:00:13 INFO - code location None
01:00:13 INFO - number of workers 2 worker options {'num_cpus': 0.5, 'memory': 2147483648, 'max_restarts': -1}
01:00:13 INFO - actor creation delay 0
01:00:13 INFO - job details {'job category': 'preprocessing', 'job name': 'text_encoder', 'job type': 'ray', 'job id': 'job_id'}
01:00:13 INFO - data factory data_ is using local data access: input_folder - output/03_chunk_out output_folder - output/04_embeddings_out
01:00:13 INFO - data factory data_ max_files -1, n_sample -1
01:00:13 INFO - data factory data_ Not using data sets, checkpointing False, max files -1, random samples -1, files to use ['.parquet'], files to checkpoint ['.parquet']
01:00:13 INFO - Running locally
2025-03-14 01:00:14,012	INFO worker.py:1777 -- Started a

✅ Stage:4 completed successfully
CPU times: user 274 ms, sys: 227 ms, total: 501 ms
Wall time: 29.1 s


### 6.2 - Inspect Generated output

In [12]:
from file_utils import read_parquet_files_as_df

input_df = read_parquet_files_as_df(output_chunk_dir)
output_df = read_parquet_files_as_df(output_embeddings_dir)

print ("Input data dimensions (rows x columns)= ", input_df.shape)
print ("Output data dimensions (rows x columns)= ", output_df.shape)

output_df.sample(min(3, output_df.shape[0]))

Successfully read 2 parquet files with 61 total rows
Successfully read 2 parquet files with 61 total rows
Input data dimensions (rows x columns)=  (61, 15)
Output data dimensions (rows x columns)=  (61, 16)


,filename,num_pages,num_tables,num_doc_elements,document_hash,ext,hash,size,date_acquired,pdf_convert_time,source_filename,removed,source_document_id,contents,document_id,embeddings
42,granite.pdf,28,21,324,3127757990743433032,pdf,96e227c7a23adc6a98590182a29641d1d8ba558847d718...,136446,2025-03-14T00:56:34.927138,172.116436,granite.pdf,[],21000182-6d62-46d3-a810-c4690afb022d,## 5 Instruction Tuning\n\nFinetuning code LLM...,90cee92e6eb1491c33f3763f77d777a5c7a5329a17aa00...,"[0.039464407, -0.03116403, 0.026932143, 0.0314..."
9,attention.pdf,15,6,436,2949302674760005271,pdf,5bbff6b2beac5f09f368a913932212e8d57953b3e3e86d...,45855,2025-03-14T00:53:42.738448,58.294557,attention.pdf,[],12bf79a1-c716-48df-aaa5-5fbc67b513c9,## 3.2.1 Scaled Dot-Product Attention\n\nWe ca...,372f585509e5743de4e6603483cad7dfa3bd5bbdfa131c...,"[-0.004859315, 0.043537527, -0.0044659376, 0.0..."
39,granite.pdf,28,21,324,3127757990743433032,pdf,96e227c7a23adc6a98590182a29641d1d8ba558847d718...,136446,2025-03-14T00:56:34.927138,172.116436,granite.pdf,[],21000182-6d62-46d3-a810-c4690afb022d,## 4.2 Training Objective\n\nFor training of a...,6ad5ff58372becb8e23633a09787fa8d2f4bcb3b6d5916...,"[0.024308162, -0.022316976, 0.07026025, 0.0556..."


## Step-7: Copy output to final output dir

In [13]:
import shutil

shutil.rmtree(MY_CONFIG.OUTPUT_FOLDER_FINAL, ignore_errors=True)
shutil.copytree(src=output_embeddings_dir, dst=MY_CONFIG.OUTPUT_FOLDER_FINAL)

print (f"✅ Copied output from '{output_embeddings_dir}' --> '{MY_CONFIG.OUTPUT_FOLDER_FINAL}'")

✅ Copied output from 'output/04_embeddings_out' --> 'output/output_final'
